In [181]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone
import os
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database
import feature_pipeline

# Import data from the DB

In [182]:
df = feature_pipeline.get_features()

In [183]:
df

,year_month_t,log_10_UNRATE_t,log_10_INDPRO_t-1,log_10_JTSJOL_t-1,UNRATE_t,INDPRO_t-1,JTSJOL_t-1,UNRATE_t+1
0,2001-01,0.623249,1.965397,3.706547,4.2,92.3415,5088.0,4.2
1,2001-02,0.623249,1.963325,3.718834,4.2,91.9020,5234.0,4.3
2,2001-03,0.633468,1.960487,3.707315,4.3,91.3034,5097.0,4.4
3,2001-04,0.643453,1.959596,3.677789,4.4,91.1162,4762.0,4.3
4,2001-05,0.633468,1.958034,3.664172,4.3,90.7891,4615.0,4.5
...,...,...,...,...,...,...,...,...
303,2026-04,0.633468,2.007553,3.838030,4.3,101.7543,6887.0,4.3
304,2026-05,0.633468,2.010808,3.879956,4.3,102.5198,7585.0,4.2
305,2026-06,0.623249,2.010766,3.877199,4.2,102.5099,7537.0,4.1
306,2026-07,0.612784,2.011937,3.856245,4.1,102.7868,7182.0,4.1


In [184]:
df[df.isna().any(axis=1)]

,year_month_t,log_10_UNRATE_t,log_10_INDPRO_t-1,log_10_JTSJOL_t-1,UNRATE_t,INDPRO_t-1,JTSJOL_t-1,UNRATE_t+1
307,2026-08,0.612784,2.012812,3.861594,4.1,102.9939,7271.0,NaN


# Splitting data into feature sets

In [185]:
df.iloc[:, 1:].corr()

,log_10_UNRATE_t,log_10_INDPRO_t-1,log_10_JTSJOL_t-1,UNRATE_t,INDPRO_t-1,JTSJOL_t-1,UNRATE_t+1
log_10_UNRATE_t,1.000000,-0.627100,-0.742547,0.984889,-0.629432,-0.688645,0.936431
log_10_INDPRO_t-1,-0.627100,1.000000,0.702608,-0.614308,0.999715,0.611356,-0.593753
log_10_JTSJOL_t-1,-0.742547,0.702608,1.000000,-0.680238,0.704636,0.977581,-0.682087
UNRATE_t,0.984889,-0.614308,-0.680238,1.000000,-0.615229,-0.618496,0.944069
INDPRO_t-1,-0.629432,0.999715,0.704636,-0.615229,1.000000,0.613945,-0.594540
JTSJOL_t-1,-0.688645,0.611356,0.977581,-0.618496,0.613945,1.000000,-0.620240
UNRATE_t+1,0.936431,-0.593753,-0.682087,0.944069,-0.594540,-0.620240,1.000000


In [186]:
df.columns[[4,5]].to_list()

['UNRATE_t', 'INDPRO_t-1']

In [187]:
sets = {
'log10_set': df.columns[1:4].to_list(),
'levels_set': df.columns[4:-1].to_list(),
'mixed_set_1': df.columns[[3,4,5]].to_list(),
'mixed_set_2': df.columns[[2,4,6]].to_list()
}

target = 'UNRATE_t+1'

In [188]:
sets

{'log10_set': ['log_10_UNRATE_t', 'log_10_INDPRO_t-1', 'log_10_JTSJOL_t-1'],
 'levels_set': ['UNRATE_t', 'INDPRO_t-1', 'JTSJOL_t-1'],
 'mixed_set_1': ['log_10_JTSJOL_t-1', 'UNRATE_t', 'INDPRO_t-1'],
 'mixed_set_2': ['log_10_INDPRO_t-1', 'UNRATE_t', 'JTSJOL_t-1']}

In [189]:
df['year_month_t'].dtype

period[M]

In [190]:
backtest_last_origin = pd.Period("2020-01")
df[df['year_month_t'] <= (backtest_last_origin)]

,year_month_t,log_10_UNRATE_t,log_10_INDPRO_t-1,log_10_JTSJOL_t-1,UNRATE_t,INDPRO_t-1,JTSJOL_t-1,UNRATE_t+1
0,2001-01,0.623249,1.965397,3.706547,4.2,92.3415,5088.0,4.2
1,2001-02,0.623249,1.963325,3.718834,4.2,91.9020,5234.0,4.3
2,2001-03,0.633468,1.960487,3.707315,4.3,91.3034,5097.0,4.4
3,2001-04,0.643453,1.959596,3.677789,4.4,91.1162,4762.0,4.3
4,2001-05,0.633468,1.958034,3.664172,4.3,90.7891,4615.0,4.5
...,...,...,...,...,...,...,...,...
224,2019-09,0.544068,2.011312,3.856003,3.5,102.6390,7178.0,3.6
225,2019-10,0.556303,2.009840,3.852724,3.6,102.2917,7124.0,3.6
226,2019-11,0.556303,2.006158,3.862668,3.6,101.4281,7289.0,3.6
227,2019-12,0.556303,2.008333,3.838093,3.6,101.9372,6888.0,3.6


In [191]:
df['year_month_t'].max()

Period('2026-08', 'M')

# Model


**“MAE-Optimized Ridge Regression Model with RMSE-tuned hyperparameter ($\lambda$)”**

In [200]:
def ridge_naive_ensemble(feature_set,
                              df,
                              n_lambda_candidates = 50,
                              n_lambda_optimization_steps = 10,
                              lower_lambda = 1e-6,
                              upper_lambda = 1e4,
                              print_details = False,
                              forecast_mode = False,
                              backtest_last_origin = None):
    # zero-variance guard
    def zero_variance_guard(std_values):
        if np.any(np.isclose(std_values, 0)):
            raise ValueError("At least one feature has zero or near-zero standard deviation.")

    # if a backtesting end date is given, create a date-capped df
    if backtest_last_origin:
        if forecast_mode:
            raise Exception(f"In backtesting mode, no forecasts can be produced!")

        backtest_last_origin = pd.Period(backtest_last_origin, freq='M')

        if backtest_last_origin >= df['year_month_t'].max():
            raise Exception(f"Your last origin backtest date has to be before the most recent observation date ({df['year_month_t'].max()})!")
        
        df = df[df['year_month_t'] <= (backtest_last_origin)].copy()

        if len(df) < 10 * len(feature_set):
            raise Exception(f"For your back test end date are not enough observations available!\nPlease choose a end date beyond {(df['year_month_t'].min() + 10 * len(feature_set)) - 1}")

        print(f"BACKTEST MODE ENABLED")
        
    # split data into training, validation, test, and target
    target = "UNRATE_t+1"
    cols = ['year_month_t'] + feature_set + ['UNRATE_t', target]
    cols = list(dict.fromkeys(cols))

    df_tvt = df[cols].dropna().copy() # tvt -> training, validation, and testing

    set_len = len(df_tvt)
    
    # in chronological order form df_tvt: 60% training and optimizing lambda and MAE weighting,
    # 60% -> 80% validation, retrain up to 80%, test against last 20% (80% -> 100%)
    train_end = int(round(set_len*0.6))
    val_end = int(round(set_len*0.8))

    X_train = df_tvt[feature_set].iloc[:train_end].to_numpy()
    X_val = df_tvt[feature_set].iloc[train_end:val_end].to_numpy()
    X_train_full = df_tvt[feature_set].iloc[:val_end].to_numpy()
    X_test = df_tvt[feature_set].iloc[val_end:].to_numpy()

    y_train = df_tvt[target].iloc[:train_end].to_numpy()
    y_val = df_tvt[target].iloc[train_end:val_end].to_numpy()
    y_train_full = df_tvt[target].iloc[:val_end].to_numpy()
    y_test = df_tvt[target].iloc[val_end:].to_numpy()

    y_pred_naive_val = df_tvt['UNRATE_t'].iloc[train_end:val_end].to_numpy()
    y_pred_naive_test = df_tvt['UNRATE_t'].iloc[val_end:].to_numpy()

    # standardize via z-score X and centering y
    def standardizing_X_centering_y(X_train, X_input, y_train):
        X_train_mean = X_train.mean(axis=0)
        X_train_std = X_train.std(axis=0)
        zero_variance_guard(X_train_std)

        Z_train = (X_train - X_train_mean) / X_train_std
        Z_input = (X_input - X_train_mean) / X_train_std

        y_train_mean = y_train.mean()

        y_train_centered = y_train - y_train_mean

        return Z_train, Z_input, y_train_mean, y_train_centered

    # fit model
    def fit_model(Z_train, Z_input, y_train_centered, y_train_mean, penalty_param, mae_w, y_pred_naive):
        A = Z_train.T @ Z_train + penalty_param * np.eye(Z_train.shape[1])
        b = Z_train.T @ y_train_centered

        beta_coeffs = np.linalg.solve(A, b)

        y_pred_raw = y_train_mean + Z_input @ beta_coeffs
        y_pred = mae_w * y_pred_raw + (1 - mae_w) * y_pred_naive

        return y_pred

    # find best lambda (penalty_param)
    # actual lambda values, not exponents -> np.geomspace is logspace for non-exponent values
    best_penalty_param = None
    best_val_rmse = np.inf

    Z_train, Z_val, y_train_mean, y_train_centered = standardizing_X_centering_y(X_train=X_train, X_input=X_val, y_train=y_train)

    for step in range(n_lambda_optimization_steps):
        lambda_space = np.geomspace(
            lower_lambda,
            upper_lambda,
            n_lambda_candidates
        )

        rmse_values = np.empty(len(lambda_space))

        for i, penalty_param in enumerate(lambda_space):
            y_pred_val = fit_model(
                Z_train=Z_train,
                Z_input=Z_val,
                y_train_centered=y_train_centered,
                y_train_mean=y_train_mean,
                penalty_param=penalty_param,
                mae_w=1,
                y_pred_naive=1
            )

            rmse = np.sqrt(np.mean((y_val - y_pred_val) ** 2))
            rmse_values[i] = rmse
        

        # pos of best lambda in current search
        best_pos = np.argmin(rmse_values)

        step_best_penalty_param = lambda_space[best_pos]
        step_best_rmse = rmse_values[best_pos]

        # store best result found overall
        if step_best_rmse < best_val_rmse:
            best_val_rmse = step_best_rmse
            best_penalty_param = step_best_penalty_param

        # refine around the best lambda
        lower_pos = max(best_pos - 1, 0)
        upper_pos = min(best_pos + 1, len(lambda_space) - 1)

        lower_lambda = lambda_space[lower_pos]
        upper_lambda = lambda_space[upper_pos]

        if print_details:
            print(
                f"Step {step + 1}: "
                f"lambda={step_best_penalty_param:.6f}, "
                f"validation RMSE={step_best_rmse:.6f}"
            )

    # refit training model with best lambda (penalty_param)
    y_pred_val = fit_model(
        Z_train=Z_train,
        Z_input=Z_val,
        y_train_centered=y_train_centered,
        y_train_mean=y_train_mean,
        penalty_param=best_penalty_param,
        mae_w=1,
        y_pred_naive=1
    )

    # find a weighting for optimal MAE due to strong performance of naive model
    weight_candidates = np.linspace(0, 1, 101)

    best_mae_w = None
    best_weighted_val_mae = np.inf

    for weight in weight_candidates:

        y_pred_val_weighted = (weight * y_pred_val + (1 - weight) * y_pred_naive_val)

        weighted_val_mae = np.mean(np.abs(y_val - y_pred_val_weighted))

        if weighted_val_mae < best_weighted_val_mae:
            best_weighted_val_mae = weighted_val_mae
            best_mae_w = weight


    # refit model on 80%
    Z_train_full, Z_test, y_train_full_mean, y_train_full_centered = standardizing_X_centering_y(X_train_full, X_test, y_train_full)

    y_pred_test = fit_model(Z_train_full,
                            Z_test,
                            y_train_full_centered,
                            y_train_full_mean,
                            best_penalty_param,
                            best_mae_w,
                            y_pred_naive_test)
    
    # --- TEST METRICS ---

    # MAE in pp
    model_mae = np.mean(np.abs(y_test - y_pred_test))
    naive_mae = np.mean(np.abs(y_test - y_pred_naive_test))

    # RMSE in pp
    model_rmse = np.sqrt(np.mean((y_test - y_pred_test) ** 2))
    naive_rmse = np.sqrt(np.mean((y_test - y_pred_naive_test) ** 2))

    # MAPE in %
    model_mape = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100
    naive_mape = np.mean(np.abs((y_test - y_pred_naive_test) / y_test)) * 100

    # model over Naive improvement in %
    if naive_mae == 0:
        model_mae_improvement = np.nan
    else:
        model_mae_improvement = (1 - (model_mae / naive_mae)) * 100
    if naive_rmse == 0:
        model_rmse_improvement = np.nan
    else:
        model_rmse_improvement = (1 - (model_rmse / naive_rmse)) * 100

    df_test_metrics = pd.DataFrame([{
        "evaluation_training_start": df_tvt['year_month_t'].iloc[0],
        "evaluation_training_end": df_tvt['year_month_t'].iloc[val_end - 1],
        "test_start": df_tvt['year_month_t'].iloc[val_end],
        "test_end": df_tvt['year_month_t'].iloc[-1],
        "lambda": np.round(best_penalty_param, 4),
        "mae_optimizing_weight": np.round(best_mae_w, 4),
        "validation_mae": np.round(best_weighted_val_mae, 4),
        "model_mae": np.round(model_mae, 4),
        "naive_mae": np.round(naive_mae, 4),
        "model_rmse": np.round(model_rmse, 4),
        "naive_rmse": np.round(naive_rmse, 4),
        "model_mae_improvement": np.round(model_mae_improvement, 4),
        "model_rmse_improvement": np.round(model_rmse_improvement, 4),
        "model_mape": np.round(model_mape, 4),
        "naive_mape": np.round(naive_mape, 4)
    }])

    # --- TEST RESULTS ---

    df_test_results = pd.DataFrame({
        "forecast_origin_t": df_tvt['year_month_t'].iloc[val_end:].copy(),
        "forecast_target_t+1": df_tvt['year_month_t'].iloc[val_end:].copy() + 1,
        "UNRATE_t+1": y_test,
        "y_pred_model_t+1": y_pred_test,
        "y_pred_naive_t+1": y_pred_naive_test,
        "model_abs_error": np.abs(y_test - y_pred_test),
        "naive_abs_error": np.abs(y_test - y_pred_naive_test),
    }).reset_index(drop=True)

    # --- FORECAST MODE ---
    df_forecast = None

    if forecast_mode:
        # use all available data
        final_train_cols = ['year_month_t'] + feature_set + [target]
        final_train_cols = list(dict.fromkeys(final_train_cols))

        forecast_cols = (['year_month_t'] + feature_set + ['UNRATE_t'])
        forecast_cols = list(dict.fromkeys(forecast_cols))

        df_final_train = df[final_train_cols].dropna().copy()
        df_forecast_input = df[forecast_cols].dropna().tail(1).copy()

        if df_forecast_input['year_month_t'].item() != df['year_month_t'].max():
            raise ValueError("The input date for the forecast is not the most up-to-date data point!")

        X_final_train = df_final_train[feature_set].to_numpy()
        y_final_train = df_final_train[target].to_numpy()

        X_forecast_input = df_forecast_input[feature_set].to_numpy()
        y_forecast_naive = df_forecast_input['UNRATE_t'].to_numpy()

        Z_final_train, Z_forecast, y_final_train_mean, y_final_train_centered = standardizing_X_centering_y(X_final_train,
                                                                                                            X_forecast_input,
                                                                                                            y_final_train)

        y_forecast = fit_model(Z_final_train,
                               Z_forecast,
                               y_final_train_centered,
                               y_final_train_mean,
                               best_penalty_param,
                               best_mae_w,
                               y_forecast_naive)

        df_forecast = pd.DataFrame({
            "production_model_training_start": df_final_train['year_month_t'].iloc[0],
            "production_model_training_end": df_final_train['year_month_t'].iloc[-1],
            "forecast_origin_t": df_forecast_input['year_month_t'].to_numpy(),
            "forecast_target_t+1": (df_forecast_input['year_month_t'] + 1).to_numpy(),
            "lambda": np.round(best_penalty_param, 4),
            "mae_optimizing_weight": np.round(best_mae_w, 4),
            "UNRATE_forecast": np.round(y_forecast, 4),
            "naive_forecast": np.round(y_forecast_naive, 4)
        })

    return df_test_metrics, df_test_results, df_forecast   

## Backtesting Mode Test

In [193]:
df_test_metrics, df_test_results, df_forecast = ridge_naive_ensemble(feature_set = sets['mixed_set_1'],
                                                                 df = df,
                                                                 n_lambda_candidates = 50,
                                                                 n_lambda_optimization_steps = 10,
                                                                 lower_lambda = 1e-6,
                                                                 upper_lambda = 1e4,
                                                                 print_details = False,
                                                                 forecast_mode = False,
                                                                 backtest_last_origin = "2021-01")

BACKTEST MODE ENABLED


In [194]:
df_test_metrics

,evaluation_training_start,evaluation_training_end,test_start,test_end,lambda,mae_optimizing_weight,validation_mae,model_mae,naive_mae,model_rmse,naive_rmse,model_mae_improvement,model_rmse_improvement,model_mape,naive_mape
0,2001-01,2017-01,2017-02,2021-01,0.162636,1.0,0.117123,0.443225,0.475,1.581525,1.59413,6.689507,0.790691,5.131255,5.377524


## Forecast Mode Test

In [195]:
df_test_metrics, df_test_results, df_forecast = ridge_naive_ensemble(feature_set = sets['mixed_set_1'],
                                                                          df = df,
                                                                 n_lambda_candidates = 50,
                                                                 n_lambda_optimization_steps = 10,
                                                                 lower_lambda = 1e-6,
                                                                 upper_lambda = 1e4,
                                                                 print_details = False,
                                                                 forecast_mode = True,
                                                                 backtest_last_origin = None)

In [196]:
df_test_metrics

,evaluation_training_start,evaluation_training_end,test_start,test_end,lambda,mae_optimizing_weight,validation_mae,model_mae,naive_mae,model_rmse,naive_rmse,model_mae_improvement,model_rmse_improvement,model_mape,naive_mape
0,2001-01,2021-06,2021-07,2026-07,4.487332,0.64,0.360892,0.089833,0.103279,0.112936,0.138197,13.018753,18.2787,2.267573,2.567951


In [197]:
df_forecast

,production_model_training_start,production_model_training_end,forecast_origin_t,forecast_target_t+1,lambda,mae_optimizing_weight,UNRATE_forecast,naive_forecast
0,2001-01,2026-07,2026-08,2026-09,4.487332,0.64,4.132956,4.1


# Feature Sets Testing

In [198]:
def feature_set_tester(keys=sets.keys(), plot_results=True):
    cols=["feature_set", "validation_mae"]
    df_feature_set_comp = pd.DataFrame(columns=cols)
    
    for key in keys:
        df_test_metrics, df_test_results, df_forecast = ridge_naive_ensemble(feature_set = sets[key],
                                                                 df = df,
                                                                 n_lambda_candidates = 50,
                                                                 n_lambda_optimization_steps = 10,
                                                                 lower_lambda = 1e-6,
                                                                 upper_lambda = 1e4,
                                                                 print_details = False,
                                                                 forecast_mode = False,
                                                                 backtest_last_origin = None)

        new_row = pd.DataFrame([
            {
                "feature_set": key,
                "validation_mae": df_test_metrics["validation_mae"].item()
            }
        ])
        df_feature_set_comp = pd.concat([df_feature_set_comp, new_row], ignore_index=True, axis=0)

        # --- ADD PLOTTING HERE ---
        if plot_results:
            fig, ax = plt.subplots(figsize=(10,5))
            ax.plot(df_test_results['forecast_target_t+1'].dt.strftime("%Y-%m"), df_test_results['UNRATE_t+1'], label=r"$y_{\text{test}}$")
            ax.plot(df_test_results['forecast_target_t+1'].dt.strftime("%Y-%m"), df_test_results['y_pred_model_t+1'], label=r"$\hat{y}_{\text{test}}$")
            ax.plot(df_test_results['forecast_target_t+1'].dt.strftime("%Y-%m"), df_test_results['y_pred_naive_t+1'], label=r"$\hat{y}_{\text{naive}}$")
            ax.tick_params("x", rotation=90, rotation_mode="xtick")
            ax.set_ylabel("UNRATE t")
            ax.set_title(f'UNRATE predict by feature set "{key}"')
            ax.legend()
            plt.tight_layout()
            plt.show()

    return df_feature_set_comp

In [199]:
feature_set_tester(plot_results=False)

,feature_set,validation_mae
0,log10_set,0.396774
1,levels_set,0.380653
2,mixed_set_1,0.360892
3,mixed_set_2,0.380329
